# 260415-1: Numpy로 RNN / LSTM / GRU Forward 구현

이번 실습에서는 세 가지 대표적인 recurrent cell 의 forward 연산을 numpy 로 직접 구현하고,
PyTorch 모듈과 비교해 검증합니다.
마지막에 세 모델의 gradient 흐름을 PyTorch autograd 로 비교해, 왜 LSTM/GRU가 등장했는지 확인합니다.

## 학습 목표
- Vanilla RNN 의 hidden state 갱신식을 이해하고 numpy 로 구현
- LSTM 의 4-gate 구조와 cell state 갱신식을 이해하고 numpy 로 구현
- GRU 의 reset/update gate 구조와 갱신식을 이해하고 numpy 로 구현
- 세 구현을 PyTorch (`nn.RNN` / `nn.LSTMCell` / `nn.GRUCell`) 와 수치 비교해 검증
- 시퀀스 길이가 길어질 때 RNN / LSTM / GRU 의 gradient 흐름이 어떻게 다른지 비교

## 구성
1. Vanilla RNN: 단일 timestep / 시퀀스 forward
2. RNN 의 long-range gradient 문제 (autograd 측정)
3. LSTM cell forward
4. GRU cell forward
5. RNN / LSTM / GRU gradient flow 비교


### Recap: 핵심 개념

#### Vanilla RNN
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$
모든 timestep 이 같은 $W_{xh}, W_{hh}, b$ 를 공유합니다. backprop 시 $W_{hh}$ 가 반복 곱해지므로, 시퀀스가 길어질수록 gradient 가 소실/폭발하기 쉽습니다.

#### LSTM (Long Short-Term Memory)
4개의 gate ($i, f, o$ + 후보 $g$) 와 cell state $c_t$ 를 두어 정보를 선택적으로 누적합니다.
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \quad h_t = o_t \odot \tanh(c_t)$$
cell state 는 곱셈/덧셈 위주로 갱신되므로 long-range gradient 흐름이 비교적 잘 유지됩니다.

#### GRU (Gated Recurrent Unit)
LSTM 의 단순화 버전으로 reset gate $r_t$ 와 update gate $z_t$ 두 개만 사용하고, cell state 와 hidden state 를 하나로 합쳤습니다.
$$\begin{aligned}
r_t &= \sigma(W_{ir} x_t + W_{hr} h_{t-1} + b_r) \\
z_t &= \sigma(W_{iz} x_t + W_{hz} h_{t-1} + b_z) \\
n_t &= \tanh(W_{in} x_t + r_t \odot (W_{hn} h_{t-1} + b_{hn}) + b_{in}) \\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1}
\end{aligned}$$
파라미터가 LSTM 보다 적어 가볍고 학습이 빠른 편입니다.

> 이번 실습에서는 backward 는 직접 작성하지 않고, **PyTorch autograd** 로 결과만 측정합니다.


In [ ]:
# Colab environment setup
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print('GPU available:')
        print(gpu_info.stdout.split('\n')[8])
    else:
        print('No GPU detected. Go to Runtime > Change runtime type > GPU')


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 6.0)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')


def rel_error(x, y):
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


## 1. Vanilla RNN

가장 기본적인 RNN cell 의 한 timestep 은 다음 한 줄입니다.

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$

- $x_t$: `(N, D)` — 미니배치, 입력 차원
- $h_{t-1}$: `(N, H)` — 미니배치, hidden 차원
- $W_{xh}$: `(D, H)`, $W_{hh}$: `(H, H)`, $b$: `(H,)`
- 결과 $h_t$: `(N, H)`

먼저 시퀀스 전체를 펼친 모습을 그려봅니다.


In [ ]:
# Unrolled RNN diagram (provided - just run)
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 5); ax.axis('off')

T = 5
x_positions = [1.5 + 2.2 * t for t in range(T)]

for t, x in enumerate(x_positions):
    ax.add_patch(FancyBboxPatch((x - 0.55, 2.1), 1.1, 0.9,
                                 boxstyle='round,pad=0.06',
                                 facecolor='#BBDEFB', edgecolor='#1565C0', lw=1.5))
    ax.text(x, 2.55, f'$h_{t+1}$', ha='center', va='center', fontsize=12, fontweight='bold')

for t, x in enumerate(x_positions):
    ax.add_patch(FancyBboxPatch((x - 0.45, 0.5), 0.9, 0.7,
                                 boxstyle='round,pad=0.05',
                                 facecolor='#FFE0B2', edgecolor='#E65100', lw=1.2))
    ax.text(x, 0.85, f'$x_{t+1}$', ha='center', va='center', fontsize=11)
    ax.annotate('', xy=(x, 2.05), xytext=(x, 1.25),
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.4))

for t in range(T - 1):
    ax.annotate('', xy=(x_positions[t + 1] - 0.6, 2.55),
                xytext=(x_positions[t] + 0.6, 2.55),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.8))
    ax.text((x_positions[t] + x_positions[t + 1]) / 2, 2.85,
            '$W_{hh}$', ha='center', fontsize=9, color='#1565C0')

ax.add_patch(FancyBboxPatch((0.05, 2.1), 0.9, 0.9,
                             boxstyle='round,pad=0.06',
                             facecolor='#E0E0E0', edgecolor='#616161', lw=1.2))
ax.text(0.5, 2.55, '$h_0$', ha='center', va='center', fontsize=11)
ax.annotate('', xy=(x_positions[0] - 0.6, 2.55), xytext=(0.95, 2.55),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.8))

ax.text(x_positions[0] + 0.2, 1.55, '$W_{xh}$', fontsize=9, color='#E65100')
ax.text(6.5, 4.3, 'Unrolled Vanilla RNN (shared $W_{xh}, W_{hh}, b$ across all timesteps)',
        ha='center', fontsize=12, fontweight='bold')
ax.text(6.5, 3.7, '$h_t = \\tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$',
        ha='center', fontsize=11, color='#333')

plt.tight_layout(); plt.show()


In [ ]:
def rnn_step_forward(x, prev_h, Wx, Wh, b):
    """
    Vanilla RNN의 단일 timestep forward 를 구현하세요.

    Args:
        x: 입력, shape (N, D)
        prev_h: 이전 hidden state, shape (N, H)
        Wx: 입력 가중치, shape (D, H)
        Wh: hidden 가중치, shape (H, H)
        b: bias, shape (H,)
    Returns:
        next_h: 다음 hidden state, shape (N, H)
        cache: backward 에서 쓰일 값들 (이번 실습에서는 사용하지 않음)

    Hint: tanh(x @ Wx + prev_h @ Wh + b)
    """
    ############################################################################
    # TODO 1: rnn_step_forward 를 구현하세요 (~2줄)                                #
    ############################################################################
    next_h = np.tanh(x @ Wx + prev_h @ Wh + b)
    cache = (x, prev_h, Wx, Wh, b, next_h)
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    return next_h, cache


In [ ]:
# Hand-checkable verification
N, D, H = 3, 10, 4
np.random.seed(0)
x = np.linspace(-0.4, 0.7, num=N * D).reshape(N, D)
prev_h = np.linspace(-0.2, 0.5, num=N * H).reshape(N, H)
Wx = np.linspace(-0.1, 0.9, num=D * H).reshape(D, H)
Wh = np.linspace(-0.3, 0.7, num=H * H).reshape(H, H)
b = np.linspace(-0.2, 0.4, num=H)

next_h, _ = rnn_step_forward(x, prev_h, Wx, Wh, b)

expected = np.array([
    [-0.58172089, -0.50182032, -0.41232771, -0.31410098],
    [ 0.66854692,  0.79562378,  0.87755553,  0.92795967],
    [ 0.97934501,  0.99144213,  0.99646691,  0.99854353],
])
print('next_h shape:', next_h.shape)
print('relative error:', rel_error(next_h, expected))
print('-> if relative error < 1e-7, the implementation is correct')


### 시퀀스 forward

길이 $T$ 입력에 대해 단일 step 을 반복적으로 적용해 모든 hidden state 를 만듭니다.
- 입력 $x$: `(N, T, D)`
- 결과 $h$: `(N, T, H)`


In [ ]:
def rnn_forward(x, h0, Wx, Wh, b):
    """
    시퀀스 길이 T 에 대해 RNN forward 를 수행하세요.

    Args:
        x: 입력 시퀀스, shape (N, T, D)
        h0: 초기 hidden state, shape (N, H)
        Wx, Wh, b: rnn_step_forward 와 동일
    Returns:
        h: 모든 timestep 의 hidden state, shape (N, T, H)
        cache: 각 step cache 의 리스트

    Hint: 결과 (N, T, H) 배열을 미리 만들고 for 문으로 채우세요.
    """
    N, T, D = x.shape
    H = h0.shape[1]
    h = np.zeros((N, T, H))
    cache = []
    ############################################################################
    # TODO 2: rnn_forward 를 구현하세요 (~5줄)                                    #
    ############################################################################
    prev_h = h0
    for t in range(T):
        next_h, step_cache = rnn_step_forward(x[:, t, :], prev_h, Wx, Wh, b)
        h[:, t, :] = next_h
        cache.append(step_cache)
        prev_h = next_h
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    return h, cache


In [ ]:
# Verify rnn_forward against PyTorch nn.RNN with copied weights
N, T, D, H = 2, 4, 5, 6
np.random.seed(42)
x_np = np.random.randn(N, T, D).astype(np.float32)
h0_np = np.zeros((N, H), dtype=np.float32)
Wx_np = (np.random.randn(D, H) * 0.3).astype(np.float32)
Wh_np = (np.random.randn(H, H) * 0.3).astype(np.float32)
b_np = (np.random.randn(H) * 0.1).astype(np.float32)

h_ours, _ = rnn_forward(x_np, h0_np, Wx_np, Wh_np, b_np)

torch_rnn = nn.RNN(input_size=D, hidden_size=H, nonlinearity='tanh', batch_first=True)
with torch.no_grad():
    torch_rnn.weight_ih_l0.copy_(torch.from_numpy(Wx_np.T))
    torch_rnn.weight_hh_l0.copy_(torch.from_numpy(Wh_np.T))
    torch_rnn.bias_ih_l0.copy_(torch.from_numpy(b_np))
    torch_rnn.bias_hh_l0.zero_()

x_torch = torch.from_numpy(x_np)
h0_torch = torch.from_numpy(h0_np).unsqueeze(0)
h_torch, _ = torch_rnn(x_torch, h0_torch)
h_torch_np = h_torch.detach().numpy()

print('our hidden states  shape:', h_ours.shape)
print('torch hidden states shape:', h_torch_np.shape)
print('relative error:', rel_error(h_ours, h_torch_np))
print('-> < 1e-5 means our numpy forward matches nn.RNN')


## 2. RNN 의 long-range gradient 문제

BPTT 식에서 $\partial L / \partial h_0$ 는 $W_{hh}$ 와 activation 미분이 $T$ 번 반복적으로 곱해진 결과입니다.
$T$ 가 커지면 이 곱이 빠르게 0 으로 수렴하거나 발산할 수 있습니다.

여기서는 backward 를 직접 작성하지 않고, **PyTorch autograd** 로 다음을 측정합니다:
1. 작은 RNN cell 을 만들고 $h_0$ 를 `requires_grad=True` 로 둠
2. 임의 입력 시퀀스를 길이 $T$ 만큼 forward
3. 마지막 hidden state 의 합을 loss 로 두고 backward
4. $\|h_0.\text{grad}\|$ 기록
5. $T \in \{2, 5, 10, 15, 20, 30\}$ 에 대해 반복

> 절대 수치는 시드/가중치/hidden 크기에 따라 달라집니다. 보고 싶은 건 **$T$ 가 커질 때 norm 이 어떤 식으로 변하는가** 입니다.


In [ ]:
# Vanishing gradient experiment for vanilla RNN (provided — just run)
def measure_rnn_h0_grad(T, H=16, D=8, seed=0):
    torch.manual_seed(seed)
    Wx = torch.randn(D, H) * 0.5
    Wh = torch.randn(H, H)
    U, S, Vh = torch.linalg.svd(Wh)
    Wh = U @ torch.diag(S / S.max()) @ Vh   # spectral radius ≈ 1
    b = torch.zeros(H)
    h0 = torch.zeros(1, H, requires_grad=True)
    x = torch.randn(T, 1, D) * 0.5
    h = h0
    for t in range(T):
        h = torch.tanh(x[t] @ Wx + h @ Wh + b)
    h.sum().backward()
    return h0.grad.norm().item()


T_list = [2, 5, 10, 15, 20, 30]
norms_rnn_only = [measure_rnn_h0_grad(T) for T in T_list]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_list, norms_rnn_only, 'o-', color='#D32F2F', lw=2, label='Vanilla RNN (tanh)')
ax.set_xlabel('Sequence length T')
ax.set_ylabel('||h0.grad||  (L2 norm)')
ax.set_yscale('log')
ax.set_title('Vanilla RNN: gradient at h0 vs sequence length')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
plt.tight_layout(); plt.show()

print('\n[ ||h0.grad|| for vanilla RNN ]')
for T, n in zip(T_list, norms_rnn_only):
    print(f'  T={T:3d}   {n:.3e}')
print('\nThe norm shrinks rapidly as T grows. This is the long-range gradient problem')
print('that motivated gated cells like LSTM and GRU.')


## 3. LSTM Cell forward

LSTM 은 cell state $c_t$ 라는 별도의 통로를 두고, gate 들로 정보를 선택적으로 갱신합니다.

먼저 hidden state 와 입력으로부터 4 개의 affine 출력을 만듭니다:

$$\begin{bmatrix} a_i \\ a_f \\ a_g \\ a_o \end{bmatrix}
= W_x x_t + W_h h_{t-1} + b$$

(PyTorch `nn.LSTMCell` 의 gate 순서가 $(i, f, g, o)$ 이므로 그대로 따릅니다.)
여기에 비선형을 씌워 4 개의 gate 를 만들고:

| 기호 | 비선형 | 역할 |
|------|--------|------|
| $i_t$ | $\sigma$ | 새 정보를 cell state 에 얼마나 쓸지 |
| $f_t$ | $\sigma$ | 이전 cell state 를 얼마나 유지할지 |
| $g_t$ | $\tanh$ | 새 정보의 후보 |
| $o_t$ | $\sigma$ | cell state 를 hidden state 로 얼마나 노출할지 |

cell / hidden state 갱신:
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)$$

### TODO 3
아래 `lstm_step_forward` 를 구현하세요.


In [ ]:
def lstm_step_forward(x, prev_h, prev_c, Wx, Wh, b):
    """
    LSTM cell 의 단일 timestep forward.

    Args:
        x: 입력, shape (N, D)
        prev_h: 이전 hidden state, shape (N, H)
        prev_c: 이전 cell state, shape (N, H)
        Wx: 입력 가중치, shape (D, 4H) — 4 gate 가 가로로 stack (i, f, g, o 순)
        Wh: hidden 가중치, shape (H, 4H)
        b: bias, shape (4H,)
    Returns:
        next_h: shape (N, H)
        next_c: shape (N, H)
        cache: (이번 실습에서는 사용하지 않음)

    Hint:
        1) a = x @ Wx + prev_h @ Wh + b      # (N, 4H)
        2) ai, af, ag, ao 를 4 등분
        3) i = sigmoid(ai), f = sigmoid(af), g = tanh(ag), o = sigmoid(ao)
        4) next_c = f * prev_c + i * g
        5) next_h = o * tanh(next_c)
    """
    N, H = prev_h.shape
    ############################################################################
    # TODO 3: lstm_step_forward 를 구현하세요 (~8줄)                                #
    ############################################################################
    a = x @ Wx + prev_h @ Wh + b
    ai = a[:, 0:H]
    af = a[:, H:2*H]
    ag = a[:, 2*H:3*H]
    ao = a[:, 3*H:4*H]
    i = sigmoid(ai)
    f = sigmoid(af)
    g = np.tanh(ag)
    o = sigmoid(ao)
    next_c = f * prev_c + i * g
    next_h = o * np.tanh(next_c)
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    cache = (x, prev_h, prev_c, Wx, Wh, b, i, f, g, o, next_c)
    return next_h, next_c, cache


In [ ]:
# Verify lstm_step_forward against PyTorch nn.LSTMCell
N, D, H = 3, 5, 4
np.random.seed(0)
x_np = np.random.randn(N, D).astype(np.float32)
prev_h_np = (np.random.randn(N, H) * 0.1).astype(np.float32)
prev_c_np = (np.random.randn(N, H) * 0.1).astype(np.float32)
Wx_np = (np.random.randn(D, 4 * H) * 0.3).astype(np.float32)
Wh_np = (np.random.randn(H, 4 * H) * 0.3).astype(np.float32)
b_np = (np.random.randn(4 * H) * 0.1).astype(np.float32)

next_h_np, next_c_np, _ = lstm_step_forward(x_np, prev_h_np, prev_c_np, Wx_np, Wh_np, b_np)

# nn.LSTMCell uses gate order (i, f, g, o) — same as ours
torch_cell = nn.LSTMCell(D, H)
with torch.no_grad():
    torch_cell.weight_ih.copy_(torch.from_numpy(Wx_np.T))
    torch_cell.weight_hh.copy_(torch.from_numpy(Wh_np.T))
    torch_cell.bias_ih.copy_(torch.from_numpy(b_np))
    torch_cell.bias_hh.zero_()

x_t = torch.from_numpy(x_np)
h_t, c_t = torch_cell(x_t, (torch.from_numpy(prev_h_np), torch.from_numpy(prev_c_np)))

print('next_h relative error:', rel_error(next_h_np, h_t.detach().numpy()))
print('next_c relative error:', rel_error(next_c_np, c_t.detach().numpy()))
print('-> < 1e-5 means our LSTM step matches nn.LSTMCell')


## 4. GRU Cell forward

GRU 는 LSTM 의 단순화 버전으로, gate 두 개 (reset $r$, update $z$) 만 사용하고 cell state 와 hidden state 를 합쳤습니다.

PyTorch `nn.GRUCell` 의 gate 순서는 $(r, z, n)$ 입니다. 식은 다음과 같습니다:

$$\begin{aligned}
r_t &= \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}) \\
z_t &= \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}) \\
n_t &= \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn})) \\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1}
\end{aligned}$$

> **주의**: PyTorch 는 input 쪽 bias `b_ih` 와 hidden 쪽 bias `b_hh` 를 따로 갖고 있어,
> reset gate 가 hidden 쪽 bias 만 곱해 들어갑니다. 검증 셀에서 이 차이를 정확히 맞춰줘야 합니다.

### TODO 4
아래 `gru_step_forward` 를 구현하세요. 위 식의 형태대로 reset/update/candidate 를 계산하면 됩니다.


In [ ]:
def gru_step_forward(x, prev_h, Wx, Wh, b_ih, b_hh):
    """
    GRU cell 의 단일 timestep forward (PyTorch nn.GRUCell 호환).

    Args:
        x: 입력, shape (N, D)
        prev_h: 이전 hidden state, shape (N, H)
        Wx: 입력 가중치, shape (D, 3H) — gate 순서 (r, z, n)
        Wh: hidden 가중치, shape (H, 3H)
        b_ih: 입력 쪽 bias, shape (3H,)
        b_hh: hidden 쪽 bias, shape (3H,)
    Returns:
        next_h: shape (N, H)
        cache

    Hint:
        gx = x @ Wx + b_ih       # (N, 3H)
        gh = prev_h @ Wh + b_hh  # (N, 3H)
        gx_r, gx_z, gx_n = 4등분 (3등분)
        gh_r, gh_z, gh_n = 마찬가지
        r = sigmoid(gx_r + gh_r)
        z = sigmoid(gx_z + gh_z)
        n = tanh(gx_n + r * gh_n)
        next_h = (1 - z) * n + z * prev_h
    """
    N, H = prev_h.shape
    ############################################################################
    # TODO 4: gru_step_forward 를 구현하세요 (~10줄)                                #
    ############################################################################
    gx = x @ Wx + b_ih
    gh = prev_h @ Wh + b_hh
    gx_r, gx_z, gx_n = gx[:, 0:H], gx[:, H:2*H], gx[:, 2*H:3*H]
    gh_r, gh_z, gh_n = gh[:, 0:H], gh[:, H:2*H], gh[:, 2*H:3*H]
    r = sigmoid(gx_r + gh_r)
    z = sigmoid(gx_z + gh_z)
    n = np.tanh(gx_n + r * gh_n)
    next_h = (1 - z) * n + z * prev_h
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    cache = (x, prev_h, r, z, n, Wx, Wh, b_ih, b_hh)
    return next_h, cache


In [ ]:
# Verify gru_step_forward against PyTorch nn.GRUCell
N, D, H = 3, 5, 4
np.random.seed(1)
x_np = np.random.randn(N, D).astype(np.float32)
prev_h_np = (np.random.randn(N, H) * 0.1).astype(np.float32)
Wx_np = (np.random.randn(D, 3 * H) * 0.3).astype(np.float32)
Wh_np = (np.random.randn(H, 3 * H) * 0.3).astype(np.float32)
b_ih_np = (np.random.randn(3 * H) * 0.1).astype(np.float32)
b_hh_np = (np.random.randn(3 * H) * 0.1).astype(np.float32)

next_h_np, _ = gru_step_forward(x_np, prev_h_np, Wx_np, Wh_np, b_ih_np, b_hh_np)

torch_cell = nn.GRUCell(D, H)
with torch.no_grad():
    torch_cell.weight_ih.copy_(torch.from_numpy(Wx_np.T))
    torch_cell.weight_hh.copy_(torch.from_numpy(Wh_np.T))
    torch_cell.bias_ih.copy_(torch.from_numpy(b_ih_np))
    torch_cell.bias_hh.copy_(torch.from_numpy(b_hh_np))

x_t = torch.from_numpy(x_np)
h_t = torch_cell(x_t, torch.from_numpy(prev_h_np))

print('next_h relative error:', rel_error(next_h_np, h_t.detach().numpy()))
print('-> < 1e-5 means our GRU step matches nn.GRUCell')


## 5. RNN / LSTM / GRU gradient flow 비교

이제 같은 시퀀스 길이에서 세 cell 의 $\|h_0.\text{grad}\|$ 를 비교합니다.
구현은 이미 검증했으니, autograd 에 맡기기 위해 PyTorch 의 `nn.RNNCell` / `nn.LSTMCell` / `nn.GRUCell` 을 사용합니다.

LSTM 은 forget-gate bias 를 1 로 두는 것이 흔한 default trick 입니다 (학습 초반 cell state 를 잊지 않도록).


In [ ]:
# Compare RNN / LSTM / GRU gradient flow via autograd
def measure_grad(cell_type, T, H=16, D=8, seed=0):
    torch.manual_seed(seed)
    if cell_type == 'rnn':
        cell = nn.RNNCell(D, H, nonlinearity='tanh')
    elif cell_type == 'lstm':
        cell = nn.LSTMCell(D, H)
        with torch.no_grad():   # forget-gate bias = 1 (common default)
            cell.bias_ih[H:2*H].fill_(1.0)
            cell.bias_hh[H:2*H].fill_(0.0)
    elif cell_type == 'gru':
        cell = nn.GRUCell(D, H)
    else:
        raise ValueError(cell_type)

    h0 = torch.zeros(1, H, requires_grad=True)
    x = torch.randn(T, 1, D) * 0.5
    if cell_type == 'lstm':
        c = torch.zeros(1, H, requires_grad=True)
        h = h0
        for t in range(T):
            h, c = cell(x[t], (h, c))
    else:
        h = h0
        for t in range(T):
            h = cell(x[t], h)
    h.sum().backward()
    return h0.grad.norm().item()


T_list = [2, 5, 10, 15, 20, 30, 50]
norms = {kind: [measure_grad(kind, T) for T in T_list] for kind in ['rnn', 'lstm', 'gru']}

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_list, norms['rnn'],  'o-', color='#D32F2F', lw=2, label='Vanilla RNN')
ax.plot(T_list, norms['lstm'], 's-', color='#1565C0', lw=2, label='LSTM')
ax.plot(T_list, norms['gru'],  '^-', color='#388E3C', lw=2, label='GRU')
ax.set_xlabel('Sequence length T')
ax.set_ylabel('||h0.grad||  (L2 norm)')
ax.set_yscale('log')
ax.set_title('Gradient at h0: RNN vs LSTM vs GRU')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
plt.tight_layout(); plt.show()

print('\n[ ||h0.grad|| ]')
print(f'  T      RNN          LSTM         GRU')
for T in T_list:
    print(f'  {T:3d}    {norms["rnn"][T_list.index(T)]:.3e}    '
          f'{norms["lstm"][T_list.index(T)]:.3e}    {norms["gru"][T_list.index(T)]:.3e}')
print('\nIn this run, vanilla RNN loses gradient very quickly while LSTM and GRU keep it')
print('much larger across the same sequence lengths.')


## 정리

| 구현한 모듈 | 핵심 식 | 검증 |
|-----------|--------|------|
| `rnn_step_forward` / `rnn_forward` | $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$ | `nn.RNN` 과 relative error < 1e-5 |
| `lstm_step_forward` | 4-gate 갱신 + cell state 통로 | `nn.LSTMCell` 과 relative error < 1e-5 |
| `gru_step_forward` | reset/update gate + 단일 hidden state | `nn.GRUCell` 과 relative error < 1e-5 |

### 정성적 결과
- 같은 시퀀스 길이에서 vanilla RNN 은 $\|h_0.\text{grad}\|$ 가 빠르게 사라지는 패턴을 보였습니다.
- LSTM 과 GRU 는 같은 길이에서도 gradient 가 훨씬 큰 값으로 유지되었습니다 — 이것이 두 구조가 등장한 핵심 이유입니다.
